# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")


✅ Libraries imported successfully!
Boto3 version: 1.37.3
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [4]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [5]:

EVENT_NAME = '202310_Hurricane_Otis'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'blackmarble_hd'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [6]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [7]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

✅ S3 client initialized successfully
   Found 68 accessible buckets
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 21 .tif files in the S3 bucket.


['drcs_activations/202310_Hurricane_Otis/blackmarble_hd/BMHD_Otis_Nov8/BMHD_VNP46A2_Otis_2023312_Nov8_Acapulco.tif',
 'drcs_activations/202310_Hurricane_Otis/blackmarble_hd/BMHD_Otis_Nov8/Cloud_VNP46A2_Otis_2023312_Nov8_Acapulco_V2.tif',
 'drcs_activations/202310_Hurricane_Otis/blackmarble_hd/BMHD_Otis_Nov9_13/BMHD_VNP46A2_Otis_2023313_Nov9_Acapulco.tif',
 'drcs_activations/202310_Hurricane_Otis/blackmarble_hd/BMHD_Otis_Nov9_13/BMHD_VNP46A2_Otis_2023314_Nov10_Acapulco.tif',
 'drcs_activations/202310_Hurricane_Otis/blackmarble_hd/BMHD_Otis_Nov9_13/BMHD_VNP46A2_Otis_2023315_Nov11_Acapulco.tif',
 'drcs_activations/202310_Hurricane_Otis/blackmarble_hd/BMHD_Otis_Nov9_13/BMHD_VNP46A2_Otis_2023316_Nov12_Acapulco.tif',
 'drcs_activations/202310_Hurricane_Otis/blackmarble_hd/BMHD_Otis_Nov9_13/BMHD_VNP46A2_Otis_2023317_Nov13_Acapulco.tif',
 'drcs_activations/202310_Hurricane_Otis/blackmarble_hd/BMHD_Otis_Nov9_13/Cloud_VNP46A2_Otis_2023313_Nov9_Acapulco_V2.tif',
 'drcs_activations/202310_Hurrican

## Configure bucket and paths (no need to create session manually)

In [8]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

## Define Chunked COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with:
- Chunked processing to handle large files
- Memory monitoring
- Progress tracking
- Proper CRS and caching

In [9]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 374
  - Total size: 50.85 GB

📁 Cached files (first 10):
  - drcs_activations/202301_Flood_CA/sentinel1/S1A_IW_20230101T015915_DVR_RTC30_G_gpufed_D7B8_WM.tif (0.6 MB)
  - drcs_activations/202301_Flood_CA/sentinel1/S1A_IW_20230101T015915_DVR_RTC30_G_gpufed_D7B8_rgb.tif (126.8 MB)
  - drcs_activations/202301_Flood_CA/sentinel1/S1A_IW_20230101T015940_DVR_RTC30_G_gpufed_839D_WM.tif (0.9 MB)
  - drcs_activations/202301_Flood_CA/sentinel1/S1A_IW_20230101T015940_DVR_RTC30_G_gpufed_839D_rgb.tif (129.6 MB)
  - drcs_activations/202301_Flood_CA/sentinel1/S1A_IW_20230101T020005_DVR_RTC30_G_gpufed_8349_WM.tif (1.0 MB)
  - drcs_activations/202301_Flood_CA/sentinel1/S1A_IW_20230101T020005_DVR_RTC30_G_gpufed_8349_rgb.tif (127.9 MB)
  - drcs_activations/202301_Flood_CA/sentinel1/S1A_IW_20230101T135955_DVR_RTC30_G_gpufed_EDD5_WM.tif (0.9 MB)
  - drcs_activations/202301_Flood_CA/sentinel1/S1A_IW_20230101T135955_DVR_RTC30_G_gpufed_EDD5_rgb.tif

(374, 54601323418)

In [10]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    _
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [11]:
keys

['drcs_activations/202310_Hurricane_Otis/blackmarble_hd/BMHD_Otis_Nov8/BMHD_VNP46A2_Otis_2023312_Nov8_Acapulco.tif',
 'drcs_activations/202310_Hurricane_Otis/blackmarble_hd/BMHD_Otis_Nov8/Cloud_VNP46A2_Otis_2023312_Nov8_Acapulco_V2.tif',
 'drcs_activations/202310_Hurricane_Otis/blackmarble_hd/BMHD_Otis_Nov9_13/BMHD_VNP46A2_Otis_2023313_Nov9_Acapulco.tif',
 'drcs_activations/202310_Hurricane_Otis/blackmarble_hd/BMHD_Otis_Nov9_13/BMHD_VNP46A2_Otis_2023314_Nov10_Acapulco.tif',
 'drcs_activations/202310_Hurricane_Otis/blackmarble_hd/BMHD_Otis_Nov9_13/BMHD_VNP46A2_Otis_2023315_Nov11_Acapulco.tif',
 'drcs_activations/202310_Hurricane_Otis/blackmarble_hd/BMHD_Otis_Nov9_13/BMHD_VNP46A2_Otis_2023316_Nov12_Acapulco.tif',
 'drcs_activations/202310_Hurricane_Otis/blackmarble_hd/BMHD_Otis_Nov9_13/BMHD_VNP46A2_Otis_2023317_Nov13_Acapulco.tif',
 'drcs_activations/202310_Hurricane_Otis/blackmarble_hd/BMHD_Otis_Nov9_13/Cloud_VNP46A2_Otis_2023313_Nov9_Acapulco_V2.tif',
 'drcs_activations/202310_Hurrican

In [13]:
# Define filename creator functions for different file types

def create_cog_filename_blackmarble_doy(f, EVENT_NAME):
    """Convert day of year (YYYYDOY) to date format and move to end of filename."""
    from datetime import datetime, timedelta
    import re
    
    filename = Path(f).stem
    extension = Path(f).suffix
    
    # Find the YYYYDOY pattern (e.g., 2023312)
    doy_pattern = r'(\d{4})(\d{3})'
    match = re.search(doy_pattern, filename)
    
    if match and len(match.group(0)) == 7:  # Ensure it's YYYYDOY format
        year = int(match.group(1))
        doy = int(match.group(2))
        
        # Convert DOY to date
        date = datetime(year, 1, 1) + timedelta(days=doy - 1)
        formatted_date = date.strftime('%Y-%m-%d')
        
        # Remove the YYYYDOY and any month/day reference from filename
        # Remove patterns like "2023312", "Nov8", "Nov9_13", etc.
        filename_clean = filename
        
        # Remove YYYYDOY
        filename_clean = re.sub(r'\d{4}\d{3}_?', '', filename_clean)
        
        # Remove month/day patterns like "Nov8", "Nov9_13", "Oct13", etc.
        filename_clean = re.sub(r'_?(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\d+(_\d+)?_?', '_', filename_clean)
        
        # Clean up multiple underscores and trailing underscores
        filename_clean = re.sub(r'_{2,}', '_', filename_clean)
        filename_clean = filename_clean.strip('_')
        
        # Create new filename
        cog_filename = f'{EVENT_NAME}_{filename_clean}_{formatted_date}_day{extension}'
    else:
        # Fallback
        cog_filename = f'{EVENT_NAME}_{filename}{extension}'
    
    return cog_filename


filter_str = 'blackmarble'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_blackmarble_doy(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202310_Hurricane_Otis_BMHD_VNP46A2_Otis_Acapulco_2023-11-08_day.tif
  202310_Hurricane_Otis_Cloud_VNP46A2_Otis_Acapulco_V2_2023-11-08_day.tif
  202310_Hurricane_Otis_BMHD_VNP46A2_Otis_Acapulco_2023-11-09_day.tif
  202310_Hurricane_Otis_BMHD_VNP46A2_Otis_Acapulco_2023-11-10_day.tif
  202310_Hurricane_Otis_BMHD_VNP46A2_Otis_Acapulco_2023-11-11_day.tif
  202310_Hurricane_Otis_BMHD_VNP46A2_Otis_Acapulco_2023-11-12_day.tif
  202310_Hurricane_Otis_BMHD_VNP46A2_Otis_Acapulco_2023-11-13_day.tif
  202310_Hurricane_Otis_Cloud_VNP46A2_Otis_Acapulco_V2_2023-11-09_day.tif
  202310_Hurricane_Otis_Cloud_VNP46A2_Otis_Acapulco_V2_2023-11-10_day.tif
  202310_Hurricane_Otis_Cloud_VNP46A2_Otis_Acapulco_V2_2023-11-11_day.tif
  202310_Hurricane_Otis_Cloud_VNP46A2_Otis_Acapulco_V2_2023-11-12_day.tif
  202310_Hurricane_Otis_Cloud_VNP46A2_Otis_Acapulco_V2_2023-11-13_day.tif
  202310_Hurricane_Otis_BMHD_VNP46A2_Otis_Acapulco_2023-10-13_day.tif
  202310_Hurricane_Otis_BMHD_VNP46A2_Otis_Aca

In [14]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_blackmarble_doy, 
                                target_dir = "Blackmarble", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202310_Hurricane_Otis_BMHD_VNP46A2_Otis_Acapulco_2023-11-08_day.tif
  202310_Hurricane_Otis_Cloud_VNP46A2_Otis_Acapulco_V2_2023-11-08_day.tif
  202310_Hurricane_Otis_BMHD_VNP46A2_Otis_Acapulco_2023-11-09_day.tif
  202310_Hurricane_Otis_BMHD_VNP46A2_Otis_Acapulco_2023-11-10_day.tif
  202310_Hurricane_Otis_BMHD_VNP46A2_Otis_Acapulco_2023-11-11_day.tif
  202310_Hurricane_Otis_BMHD_VNP46A2_Otis_Acapulco_2023-11-12_day.tif
  202310_Hurricane_Otis_BMHD_VNP46A2_Otis_Acapulco_2023-11-13_day.tif
  202310_Hurricane_Otis_Cloud_VNP46A2_Otis_Acapulco_V2_2023-11-09_day.tif
  202310_Hurricane_Otis_Cloud_VNP46A2_Otis_Acapulco_V2_2023-11-10_day.tif
  202310_Hurricane_Otis_Cloud_VNP46A2_Otis_Acapulco_V2_2023-11-11_day.tif
  202310_Hurricane_Otis_Cloud_VNP46A2_Otis_Acapulco_V2_2023-11-12_day.tif
  202310_Hurricane_Otis_Cloud_VNP46A2_Otis_Acapulco_V2_2023-11-13_day.tif
  202310_Hurricane_Otis_BMHD_VNP46A2_Otis_Acapulco_2023-10-13_day.tif
  202310_Hurricane_Otis_BMHD_VNP46A2_Otis_Acapu

Reading input: /tmp/tmp58gvaio9_temp.tif



   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmptw2q6w69.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Blackmarble/202310_Hurricane_Otis_BMHD_VNP46A2_Otis_Acapulco_2023-11-08_day.tif
   [MEMORY] Final: 456.8 MB (Change: +161.7 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202310_Hurricane_Otis_BMHD_VNP46A2_Otis_Acapulco_2023-11-08_day.tif

[2/21] Processing: drcs_activations/202310_Hurricane_Otis/blackmarble_hd/BMHD_Otis_Nov8/Cloud_VNP46A2_Otis_2023312_Nov8_Acapulco_V2.tif
   Output filename: 202310_Hurricane_Otis_Cloud_VNP46A2_Otis_Acapulco_V2_2023-11-08_day.tif


...

✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202310_Hurricane_Otis_Cloud_VNP46A2_Otis_Acapulco_V2_2023-10-29_day.tif

[19/21] Processing: drcs_activations/202310_Hurricane_Otis/blackmarble_hd/Cloud_VNP46A2_Otis_2023307_Nov2_Acapulco_V2.tif
   Output filename: 202310_Hurricane_Otis_Cloud_VNP46A2_Otis_Acapulco_V2_2023-11-03_day.tif
   [MEMORY] Initial: 442.7 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=1, center sample non-zero=26/5625
            Estimated data coverage: 0.4% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to

Reading input: /tmp/tmp1m3o0k54_temp.tif

Updating dataset tags...
Writing output to: /tmp/tmpnb85sa3l.tif


✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202310_Hurricane_Otis_Cloud_VNP46A2_Otis_Acapulco_V2_2023-11-03_day.tif

[20/21] Processing: drcs_activations/202310_Hurricane_Otis/blackmarble_hd/Cloud_VNP46A2_Otis_2023308_Nov3_Acapulco_V2.tif
   Output filename: 202310_Hurricane_Otis_Cloud_VNP46A2_Otis_Acapulco_V2_2023-11-04_day.tif
   [MEMORY] Initial: 444.5 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=1, center sample non-zero=30/5625
            Estimated data coverage: 0.4% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to

Reading input: /tmp/tmp6laq4001_temp.tif

Updating dataset tags...
Writing output to: /tmp/tmpzx0lbn_5.tif


✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202310_Hurricane_Otis_Cloud_VNP46A2_Otis_Acapulco_V2_2023-11-04_day.tif

[21/21] Processing: drcs_activations/202310_Hurricane_Otis/blackmarble_hd/Cloud_VNP46A2_Otis_2023309_Nov4_Acapulco_V2.tif
   Output filename: 202310_Hurricane_Otis_Cloud_VNP46A2_Otis_Acapulco_V2_2023-11-05_day.tif
   [MEMORY] Initial: 444.5 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=1, center sample non-zero=38/5625
            Estimated data coverage: 1.3% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to

## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [ ]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")